# Semantic search with FAISS (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
# Install required libraries including faiss-gpu.
# faiss-gpu = Facebook AI Similarity Search (GPU version) for fast vector search.
# Use 'faiss-cpu' if you don't have a GPU.

!pip install datasets evaluate transformers[sentencepiece]
!pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


In [2]:
# Load the GitHub issues dataset we created in the previous notebook.
# This dataset has 2855 rows with fields: url, title, body, comments, is_pull_request, etc.

from datasets import load_dataset

issues_dataset = load_dataset("lewtun/github-issues", split="train")
issues_dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


datasets-issues-with-comments.jsonl:   0%|          | 0.00/12.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3019 [00:00<?, ? examples/s]

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'timeline_url', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 3019
})

In [3]:
# Filter to keep only real issues (not PRs) that have at least 1 comment.
# PRs tend not to contain useful Q&A information for a search engine.
# Issues with no comments have no answers to return — useless for search.
# This reduces 2855 → 771 rows.

issues_dataset = issues_dataset.filter(
    lambda x: (x["is_pull_request"] == False and len(x["comments"]) > 0)
)
issues_dataset

Filter:   0%|          | 0/3019 [00:00<?, ? examples/s]

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'timeline_url', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 808
})

In [4]:
# Keep only the 4 columns we need for building the search engine.
# symmetric_difference() finds columns that are in one set but not both,
# giving us the list of columns TO REMOVE (everything except our 4 keepers).

columns = issues_dataset.column_names
columns_to_keep = ["title", "body", "html_url", "comments"]
columns_to_remove = set(columns_to_keep).symmetric_difference(columns)
issues_dataset = issues_dataset.remove_columns(columns_to_remove)
issues_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 808
})

In [5]:
# Switch to Pandas format and load the full dataset into a DataFrame.
# We need Pandas here to use .explode() in the next step.
# df = issues_dataset[:] loads everything at once into a DataFrame.

issues_dataset.set_format("pandas")
df = issues_dataset[:]

In [6]:
# Preview the comments for the first issue.
# Each issue has a list of comments — we need to 'explode' these
# so each comment becomes its own row (for individual embedding).

df["comments"][0].tolist()

['Cool, I think we can do both :)',
 '@lhoestq now the 2 are implemented.\r\n\r\nPlease note that for the the second protection, finally I have chosen to protect the master branch only from **merge commits** (see update comment above), so no need to disable/re-enable the protection on each release (direct commits, different from merge commits, can be pushed to the remote master branch; and eventually reverted without messing up the repo history).']

In [7]:
# Explode the comments list so each comment gets its own row.
# Before: 1 row per issue, comments = [comment1, comment2, ...]
# After: 1 row per comment, with title/body/url duplicated for each.
# Example: issue with 4 comments → 4 rows in comments_df.

comments_df = df.explode("comments", ignore_index=True)
comments_df.head(4)

,html_url,title,comments,body
0,https://github.com/huggingface/datasets/issues...,Protect master branch,"Cool, I think we can do both :)",After accidental merge commit (91c55355b634d0d...
1,https://github.com/huggingface/datasets/issues...,Protect master branch,@lhoestq now the 2 are implemented.\r\n\r\nPle...,After accidental merge commit (91c55355b634d0d...
2,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,Hi ! I guess the caching mechanism should have...,## Describe the bug\r\nAfter upgrading to data...
3,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,"If it's easy enough to implement, then yes ple...",## Describe the bug\r\nAfter upgrading to data...


In [8]:
# Convert the exploded Pandas DataFrame back to a HuggingFace Dataset.
# Dataset.from_pandas() wraps any DataFrame.
# Now we have 2842 individual comment rows to build embeddings for.

from datasets import Dataset

comments_dataset = Dataset.from_pandas(comments_df)
comments_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 2964
})

In [9]:
# Add a 'comment_length' column (word count for each comment).
# .split() splits on whitespace to count words.
# We'll use this to filter out trivially short comments.

comments_dataset = comments_dataset.map(
    lambda x: {"comment_length": len(x["comments"].split())}
)

Map:   0%|          | 0/2964 [00:00<?, ? examples/s]

In [10]:
# Filter out comments with 15 or fewer words.
# Short comments like 'cc @user' or 'Thanks!' add noise to search results.
# This reduces 2842 → 2098 comments.

comments_dataset = comments_dataset.filter(lambda x: x["comment_length"] > 15)
comments_dataset

Filter:   0%|          | 0/2964 [00:00<?, ? examples/s]

Dataset({
    features: ['html_url', 'title', 'comments', 'body', 'comment_length'],
    num_rows: 2175
})

In [11]:
# Create a 'text' column by concatenating title + body + comment.
# This gives the embedding model maximum context — it sees the full issue thread.
# The '\n' separators help the model distinguish the three sections.

def concatenate_text(examples):
    return {
        "text": examples["title"]
        + " \n "
        + examples["body"]
        + " \n "
        + examples["comments"]
    }


comments_dataset = comments_dataset.map(concatenate_text)

Map:   0%|          | 0/2175 [00:00<?, ? examples/s]

In [12]:
# Load the embedding model and tokenizer.
# 'multi-qa-mpnet-base-dot-v1' is designed specifically for semantic search.
# It was trained to match questions with relevant passages.

from transformers import AutoTokenizer, AutoModel

model_ckpt = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
# Move the model to GPU for faster embedding computation.
# torch.device('cuda') selects the GPU.
# .to(device) transfers model weights to GPU memory.

import torch

device = torch.device("cuda")
model.to(device)

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

In [14]:
# Define CLS pooling to get a single vector per text.
# Transformers produce one embedding per token — we need one per text.
# The [CLS] token (index 0) is a special summary token trained to represent the whole input.
# We extract its embedding: shape [batch_size, 768].

def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]

In [15]:
# Define get_embeddings() — our main embedding function.
# 1. Tokenize the text (padding & truncation to handle variable lengths).
# 2. Move input tensors to GPU.
# 3. Run the model forward pass.
# 4. Apply CLS pooling to get one 768-dim vector per text.

def get_embeddings(text_list):
    encoded_input = tokenizer(
        text_list, padding=True, truncation=True, return_tensors="pt"
    )
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    model_output = model(**encoded_input)
    return cls_pooling(model_output)

In [16]:
# Test get_embeddings() on one example.
# Output shape [1, 768] = 1 text → 768-dimensional vector.
# This 768-dim vector is a numerical 'fingerprint' of the text's meaning.

embedding = get_embeddings(comments_dataset["text"][0])
embedding.shape

torch.Size([1, 768])

In [17]:
# Generate embeddings for ALL 2098 comments and store them in the dataset.
# .detach() removes gradient tracking (we don't need backprop).
# .cpu() moves tensor back from GPU to CPU.
# .numpy()[0] converts to a NumPy array (required by FAISS).

embeddings_dataset = comments_dataset.map(
    lambda x: {"embeddings": get_embeddings(x["text"]).detach().cpu().numpy()[0]}
)

Map:   0%|          | 0/2175 [00:00<?, ? examples/s]

In [19]:
pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 90.7 MB/s eta 0:00:00


In [22]:
pip install faiss-gpu-cu12


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 12.0 MB/s eta 0:00:00


In [23]:
# Build a FAISS index over all the embeddings.
# .add_faiss_index() creates a fast nearest-neighbour search index.
# Internally it uses an IVF (Inverted File) or flat index structure.
# Once built, we can search millions of vectors in milliseconds.

embeddings_dataset.add_faiss_index(column="embeddings")

ImportError: You must install Faiss to use FaissIndex. To do so you can run `conda install -c pytorch faiss-cpu` or `conda install -c pytorch faiss-gpu`. A community supported package is also available on pypi: `pip install faiss-cpu` or `pip install faiss-gpu`. Note that pip may not have the latest version of FAISS, and thus, some of the latest features and bug fixes may not be available.

In [ ]:
# Embed the search query using the SAME model.
# Crucial: query and documents MUST use the same embedding model!
# .cpu().detach().numpy() converts the GPU tensor to a NumPy array for FAISS.

question = "How can I load a dataset offline?"
question_embedding = get_embeddings([question]).cpu().detach().numpy()
question_embedding.shape

In [ ]:
# Search the FAISS index for the 5 nearest neighbours to our query.
# 'k=5' = return top 5 most similar comments.
# Returns: scores (similarity values) and samples (the matching dataset rows).

scores, samples = embeddings_dataset.get_nearest_examples(
    "embeddings", question_embedding, k=5
)

In [ ]:
# Collect search results into a Pandas DataFrame for easy viewing.
# Attach the similarity scores as a new column.
# Sort descending so the best match appears first.

import pandas as pd

samples_df = pd.DataFrame.from_dict(samples)
samples_df["scores"] = scores
samples_df.sort_values("scores", ascending=False, inplace=True)

In [ ]:
# Print the top 5 search results.
# Each result shows the matched comment, its similarity score, issue title, and URL.
# Notice that all top results are from the same issue — they all discuss offline mode!
# High scores (22–25) indicate strong semantic similarity to our query.

for _, row in samples_df.iterrows():
    print(f"COMMENT: {row.comments}")
    print(f"SCORE: {row.scores}")
    print(f"TITLE: {row.title}")
    print(f"URL: {row.html_url}")
    print("=" * 50)
    print()